In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast

In [2]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [3]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

In [4]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [5]:
#function to split words and sentences into two different dataframes (currently doesn't used)
def split_words_sentences_short(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [6]:
#function to split words and sentences into two different dataframes for singular/plural dataframes
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [7]:
# read csv file
df_rules = pd.read_csv("rules_20JUN_German_h.csv")

In [8]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2234 entries, 0 to 2233
Data columns (total 16 columns):
Lemma                             2233 non-null object
Word_Type                         2233 non-null object
Priority                          277 non-null object
Category                          2233 non-null object
Primary_subcategory               2233 non-null object
Secondary_subcategory             812 non-null object
Alt_Field                         880 non-null object
Alt_Sg_Replacement                988 non-null object
Alt_Sg_/_and_inclusive_form       294 non-null object
Alt_Sg_people_first               251 non-null object
Alt_Sg_reframed                   213 non-null object
Alt_Pl_collective_noun            48 non-null object
Alt_Pl                            173 non-null object
Alt_Pl_pair_and_inclusive_form    251 non-null object
Alt_Pl_people_first               146 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

In [9]:
#drop row if all value in row are missing
df_rules.dropna(how='all', inplace=True)

In [10]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2233 entries, 0 to 2233
Data columns (total 16 columns):
Lemma                             2233 non-null object
Word_Type                         2233 non-null object
Priority                          277 non-null object
Category                          2233 non-null object
Primary_subcategory               2233 non-null object
Secondary_subcategory             812 non-null object
Alt_Field                         880 non-null object
Alt_Sg_Replacement                988 non-null object
Alt_Sg_/_and_inclusive_form       294 non-null object
Alt_Sg_people_first               251 non-null object
Alt_Sg_reframed                   213 non-null object
Alt_Pl_collective_noun            48 non-null object
Alt_Pl                            173 non-null object
Alt_Pl_pair_and_inclusive_form    251 non-null object
Alt_Pl_people_first               146 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

### Drop duplicates

In [11]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives


In [12]:
#get index of duplicates
i1 = df_rules[((df_rules.Lemma == 'Ally'))].index

In [13]:
#get index of duplicates
i2 = df_rules[((df_rules.Lemma == 'LGBTQIA+'))].index

In [14]:
#get index of duplicates
i3 = df_rules[((df_rules.Lemma == 'Zwischengeschlechtlichkeit'))].index

In [15]:
#drop row with one duplicate
#df_rules = df_rules.drop(i1[1])

In [16]:
#drop row with one duplicate
#df_rules = df_rules.drop(i2[0])

In [17]:
#drop row with one duplicate
#df_rules = df_rules.drop(i3[0])

In [18]:
df_rules["Lemma"].duplicated().sum()

0

In [19]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2233 entries, 0 to 2233
Data columns (total 16 columns):
Lemma                             2233 non-null object
Word_Type                         2233 non-null object
Priority                          277 non-null object
Category                          2233 non-null object
Primary_subcategory               2233 non-null object
Secondary_subcategory             812 non-null object
Alt_Field                         880 non-null object
Alt_Sg_Replacement                988 non-null object
Alt_Sg_/_and_inclusive_form       294 non-null object
Alt_Sg_people_first               251 non-null object
Alt_Sg_reframed                   213 non-null object
Alt_Pl_collective_noun            48 non-null object
Alt_Pl                            173 non-null object
Alt_Pl_pair_and_inclusive_form    251 non-null object
Alt_Pl_people_first               146 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing

In [20]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives


In [21]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [22]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2233 entries, 0 to 2233
Data columns (total 16 columns):
Lemma                             2233 non-null object
Word_Type                         2233 non-null object
Priority                          277 non-null object
Category                          2233 non-null object
Primary_subcategory               2233 non-null object
Secondary_subcategory             812 non-null object
Alt_Field                         880 non-null object
Alt_Sg_Replacement                988 non-null object
Alt_Sg_/_and_inclusive_form       294 non-null object
Alt_Sg_people_first               251 non-null object
Alt_Sg_reframed                   213 non-null object
Alt_Pl_collective_noun            48 non-null object
Alt_Pl                            173 non-null object
Alt_Pl_pair_and_inclusive_form    251 non-null object
Alt_Pl_people_first               146 non-null object
3rd_party_alternatives            3 non-null object
dtypes: object(16)
memory us

## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [23]:
#order for plural form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Alt_Sg_Replacement", "Alt_Field", "Alt_Sg_/_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [24]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [25]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Alt_Pl", "Alt_Pl_pair_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [26]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [27]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [28]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives'] = df_rules.iloc[:,source_col_loc:source_col_loc+9].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)


In [29]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [30]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Alt_Sg_people_first,...,Alt_Pl,Alt_Pl_pair_and_inclusive_form,Alt_Pl_people_first,3rd_party_alternatives,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service ((x)aaS) | etwas als Di...,NaN,NaN,...,NaN,NaN,NaN,NaN,something as a service ((x)aaS) | etwas als Di...,"[something as a service ((x)aaS), etwas als Di...",,[],something as a service ((x)aaS) | etwas als Di...,"[something as a service ((x)aaS), etwas als Di..."
1,ACCT,abbr,NaN,style,abbreviation,finance,NaN,Account | Konto,NaN,NaN,...,NaN,NaN,NaN,NaN,Account | Konto,"[Account, Konto]",,[],Account | Konto,"[Account, Konto]"
2,ADD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Disorder | Attention Deficit...,NaN,NaN,...,NaN,NaN,NaN,NaN,Attention Deficit Disorder | Attention Deficit...,"[Attention Deficit Disorder, Attention Deficit...",,[],Attention Deficit Disorder | Attention Deficit...,"[Attention Deficit Disorder, Attention Deficit..."
3,ADHD,acr,NaN,style,abbreviation,medical,NaN,Attention Deficit Hyperactivity Disorder | Att...,NaN,NaN,...,NaN,NaN,NaN,NaN,Attention Deficit Hyperactivity Disorder | Att...,"[Attention Deficit Hyperactivity Disorder, Att...",,[],Attention Deficit Hyperactivity Disorder | Att...,"[Attention Deficit Hyperactivity Disorder, Att..."
4,ADHS,acr,NaN,style,abbreviation,medical,NaN,Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,NaN,NaN,...,NaN,NaN,NaN,NaN,Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,[Aufmerksamkeitsdefizit-Hyperaktivitätsstörung...,,[],Aufmerksamkeitsdefizit-Hyperaktivitätsstörung ...,[Aufmerksamkeitsdefizit-Hyperaktivitätsstörung...


### Create subsets of dataframes

In [31]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"]]

### Check duplicates&homonyms

In [32]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


## Acronyms (abbreviations)

In [33]:
df_rules["Word_Type"].unique()

array(['acr', 'abbr', 's', 'S', 'i', 'a', 's+v', 'v', 'adv+a', 'v+s',
       'adv', 'a+s', 'a+v'], dtype=object)

In [34]:
df_abbr = df_rules[(df_rules["Word_Type"]=='abbr')]

In [35]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 34 entries, 1 to 2093
Data columns (total 8 columns):
Lemma                  34 non-null object
Word_Type              34 non-null object
Category               34 non-null object
Primary_subcategory    34 non-null object
Sg_all_split           34 non-null object
Pl_all                 34 non-null object
Pl_all_split           34 non-null object
Alt_split              34 non-null object
dtypes: object(8)
memory usage: 2.4+ KB


In [36]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [37]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations_DE.csv", index=False)

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [38]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [39]:
df_rules = df_rules.query("Lemma != @abbr_list")

In [40]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2199 entries, 0 to 2233
Data columns (total 8 columns):
Lemma                  2199 non-null object
Word_Type              2199 non-null object
Category               2199 non-null object
Primary_subcategory    2199 non-null object
Sg_all_split           2199 non-null object
Pl_all                 2199 non-null object
Pl_all_split           2199 non-null object
Alt_split              2199 non-null object
dtypes: object(8)
memory usage: 154.6+ KB


## Openly_discriminating

In [41]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [42]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 385 entries, 23 to 2171
Data columns (total 8 columns):
Lemma                  385 non-null object
Word_Type              385 non-null object
Category               385 non-null object
Primary_subcategory    385 non-null object
Sg_all_split           385 non-null object
Pl_all                 385 non-null object
Pl_all_split           385 non-null object
Alt_split              385 non-null object
dtypes: object(8)
memory usage: 27.1+ KB


In [43]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
23,Abwichshure,s,openly_discriminating,sexism,[-],-,[-],[-]
26,Affenfotze,s,openly_discriminating,sexism,[-],-,[-],[-]
28,Albino,s,openly_discriminating,ableism,[-],-,[-],[-]
29,Allerweltshure,s,openly_discriminating,sexism,[-],-,[-],[-]
31,Allmannshure,s,openly_discriminating,sexism,[-],-,[-],[-]


In [44]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [45]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Abwichshure,s,openly_discriminating,sexism,[-],-,[-],[-]
1,Affenfotze,s,openly_discriminating,sexism,[-],-,[-],[-]
2,Albino,s,openly_discriminating,ableism,[-],-,[-],[-]
3,Allerweltshure,s,openly_discriminating,sexism,[-],-,[-],[-]
4,Allmannshure,s,openly_discriminating,sexism,[-],-,[-],[-]


In [46]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [47]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words_DE.csv", index=False)

In [48]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences_DE.csv", index=False)

## Style

In [49]:
df_style = df_rules[df_rules["Category"]=="style"]

In [50]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 679 entries, 0 to 2233
Data columns (total 8 columns):
Lemma                  679 non-null object
Word_Type              679 non-null object
Category               679 non-null object
Primary_subcategory    679 non-null object
Sg_all_split           679 non-null object
Pl_all                 679 non-null object
Pl_all_split           679 non-null object
Alt_split              679 non-null object
dtypes: object(8)
memory usage: 47.7+ KB


In [51]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [52]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [53]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words_DE.csv", index=False)

In [54]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences_DE.csv", index=False)

## Inclusive

In [55]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [56]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 323 entries, 22 to 2216
Data columns (total 8 columns):
Lemma                  323 non-null object
Word_Type              323 non-null object
Category               323 non-null object
Primary_subcategory    323 non-null object
Sg_all_split           323 non-null object
Pl_all                 323 non-null object
Pl_all_split           323 non-null object
Alt_split              323 non-null object
dtypes: object(8)
memory usage: 22.7+ KB


In [57]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
22,Abhängigkeit,s,inclusive,communal,[],,[],[]
24,Achtsamkeit,s,inclusive,communal,[],,[],[]
30,Allies,s,inclusive,d_and_i,[Unterstützer~innen~/~Unterstützer],,[],[Unterstützer~innen~/~Unterstützer]
32,Allonormativität,s,inclusive,d_and_i,[],,[],[]
33,Ally,s,inclusive,d_and_i,"[Unterstützer~in, Unterstützer~innen]",Unterstützer~innen,[Unterstützer~innen],"[Unterstützer~in, Unterstützer~innen]"


In [58]:
df_inclusive["Primary_subcategory"].unique()

array(['communal', 'd_and_i'], dtype=object)

In [59]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [60]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [61]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 4 columns):
Lemma                  17 non-null object
Word_Type              17 non-null object
Category               17 non-null object
Primary_subcategory    17 non-null object
dtypes: object(4)
memory usage: 672.0+ bytes


In [62]:
df_inclusive_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 4 columns):
Lemma                  306 non-null object
Word_Type              306 non-null object
Category               306 non-null object
Primary_subcategory    306 non-null object
dtypes: object(4)
memory usage: 9.7+ KB


In [63]:
df_communal_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='communal']

In [64]:
df_communal_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 217 entries, 0 to 305
Data columns (total 4 columns):
Lemma                  217 non-null object
Word_Type              217 non-null object
Category               217 non-null object
Primary_subcategory    217 non-null object
dtypes: object(4)
memory usage: 8.5+ KB


In [65]:
df_communal_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='communal']

In [66]:
df_communal_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 4 columns):
Lemma                  0 non-null object
Word_Type              0 non-null object
Category               0 non-null object
Primary_subcategory    0 non-null object
dtypes: object(4)
memory usage: 0.0+ bytes


In [67]:
#save to csv file after preprocessing
df_communal_word.to_csv("communal_DE.csv", index=False)

In [68]:
df_d_and_i_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='d_and_i']

In [69]:
df_d_and_i_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 89 entries, 2 to 288
Data columns (total 4 columns):
Lemma                  89 non-null object
Word_Type              89 non-null object
Category               89 non-null object
Primary_subcategory    89 non-null object
dtypes: object(4)
memory usage: 3.5+ KB


In [70]:
df_d_and_i_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='d_and_i']

In [71]:
df_d_and_i_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 17 entries, 0 to 16
Data columns (total 4 columns):
Lemma                  17 non-null object
Word_Type              17 non-null object
Category               17 non-null object
Primary_subcategory    17 non-null object
dtypes: object(4)
memory usage: 680.0+ bytes


In [72]:
#save to csv file after preprocessing
df_d_and_i_word.to_csv("d_and_i_words_DE.csv", index=False)

In [73]:
#save to csv file after preprocessing
df_d_and_i_sentence.to_csv("d_and_i_sentences_DE.csv", index=False)

In [74]:
text = "Mikroagression"

In [75]:
tokens = model["de"](text)

In [76]:
for token in tokens:
    print (token.text, token.lemma_)

Mikroagression Mikroagression


## Gendered

In [77]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [78]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 312 entries, 20 to 2219
Data columns (total 8 columns):
Lemma                  312 non-null object
Word_Type              312 non-null object
Category               312 non-null object
Primary_subcategory    312 non-null object
Sg_all_split           312 non-null object
Pl_all                 312 non-null object
Pl_all_split           312 non-null object
Alt_split              312 non-null object
dtypes: object(8)
memory usage: 21.9+ KB


In [79]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_words_sentences(df_gendered)

In [80]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 292 entries, 0 to 291
Data columns (total 8 columns):
Lemma                  292 non-null object
Word_Type              292 non-null object
Category               292 non-null object
Primary_subcategory    292 non-null object
Sg_all_split           292 non-null object
Pl_all                 292 non-null object
Pl_all_split           292 non-null object
Alt_split              292 non-null object
dtypes: object(8)
memory usage: 18.4+ KB


In [81]:
df_gendered_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
Lemma                  20 non-null object
Word_Type              20 non-null object
Category               20 non-null object
Primary_subcategory    20 non-null object
Sg_all_split           20 non-null object
Pl_all                 20 non-null object
Pl_all_split           20 non-null object
Alt_split              20 non-null object
dtypes: object(8)
memory usage: 1.4+ KB


In [82]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's'")

In [83]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 285 entries, 0 to 288
Data columns (total 8 columns):
Lemma                  285 non-null object
Word_Type              285 non-null object
Category               285 non-null object
Primary_subcategory    285 non-null object
Sg_all_split           285 non-null object
Pl_all                 285 non-null object
Pl_all_split           285 non-null object
Alt_split              285 non-null object
dtypes: object(8)
memory usage: 20.0+ KB


In [84]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [85]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all"], axis=1)

In [86]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words_DE.csv", index=False)

In [87]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words_DE.csv", index=False)

In [88]:
df_gendered_sentence.to_csv("gendered_sentences_DE.csv", index=False)

## Unconscious Bias

In [89]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [90]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 500 entries, 53 to 2232
Data columns (total 8 columns):
Lemma                  500 non-null object
Word_Type              500 non-null object
Category               500 non-null object
Primary_subcategory    500 non-null object
Sg_all_split           500 non-null object
Pl_all                 500 non-null object
Pl_all_split           500 non-null object
Alt_split              500 non-null object
dtypes: object(8)
memory usage: 35.2+ KB


In [91]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences(df_ub)

In [92]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401 entries, 0 to 400
Data columns (total 8 columns):
Lemma                  401 non-null object
Word_Type              401 non-null object
Category               401 non-null object
Primary_subcategory    401 non-null object
Sg_all_split           401 non-null object
Pl_all                 401 non-null object
Pl_all_split           401 non-null object
Alt_split              401 non-null object
dtypes: object(8)
memory usage: 25.2+ KB


In [93]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [94]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 44 entries, 0 to 203
Data columns (total 8 columns):
Lemma                  44 non-null object
Word_Type              44 non-null object
Category               44 non-null object
Primary_subcategory    44 non-null object
Sg_all_split           44 non-null object
Pl_all                 44 non-null object
Pl_all_split           44 non-null object
Alt_split              44 non-null object
dtypes: object(8)
memory usage: 3.1+ KB


#### save the left data into separate dataframe

In [95]:
plural_list = list(df_ub_word_pl["Lemma"])

In [96]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all" 

In [97]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all"], axis=1)

#### save to csv files after preprocessing

In [98]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words_DE.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words_DE.csv", index=False)
df_ub_sentence.to_csv("ub_sentences_DE.csv", index=False)